In [ ]:
import gzip
import json
import os
from datetime import datetime
from io import StringIO
from pathlib import Path


import pandas as pd
import requests
from tqdm import tqdm

from kibad_crawler.config import RAW_DATA_DIR, S2_API_KEY, MAX_NUM_BULK_PAPERS
from kibad_crawler import s2_api

# Übersicht relevante Journals und Archive

Download [Übersicht relevante Journals und Archive](https://docs.google.com/spreadsheets/d/1ugPuiRNmcVYWcV98TvXsyqVwJYhmk2iZjZ4o9estM2Y/edit?gid=1647072097#gid=1647072097) as CSV and save it to `RAW_DATA_DIR`

In [ ]:
# https://docs.google.com/spreadsheets/d/1ugPuiRNmcVYWcV98TvXsyqVwJYhmk2iZjZ4o9estM2Y/edit?gid=1647072097#gid=1647072097

df_journals = pd.read_csv(
    RAW_DATA_DIR
    / "Übersicht relevante Journals und Archive - Ergebnis systematische Literatursuche.csv"
)
issn = [x for x in df_journals["Online ISSN"].unique() if isinstance(x, str)] + [
    x for x in df_journals["Print ISSN"].unique() if isinstance(x, str)
]
print(f"{len(issn)} ISSN codes")

# Semantic Scholar

In [ ]:
def download_dataset(release_id: str, dataset_name: str) -> requests.Response:
    url = (
        f"https://api.semanticscholar.org/datasets/v1/release/{release_id}/dataset/{dataset_name}"
    )

    headers = {"x-api-key": S2_API_KEY}
    response = requests.get(url, headers=headers)
    return response


def download_file(url: str, path: Path, chunk_size: int = 10 * 1024):
    path.mkdir(parents=True, exist_ok=True)
    file_name = url.split("?")[0].split("/")[-1]
    with requests.get(url, stream=True, allow_redirects=True) as response:
        with open(path / file_name, mode="wb") as file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                file.write(chunk)
    return path / file_name

## Release

In [ ]:
response_releases = requests.get("https://api.semanticscholar.org/datasets/v1/release").json()
latest_release = response_releases[-1]
print(f"{response_releases[-1]} latest release")

In [ ]:
# DATA_DIR = Path("/Users/darenasc/projects/dfki/kiba-d/data")

release_path = RAW_DATA_DIR / "semantic_scholar" / "release" / f"{latest_release}"
release_path.mkdir(exist_ok=True, parents=True)

In [ ]:
response_datasets = requests.get(
    f"https://api.semanticscholar.org/datasets/v1/release/{latest_release}"
).json()

In [ ]:
datasets = pd.DataFrame(response_datasets["datasets"])
datasets

## Publication Venues

In [ ]:
# `publication-venues`
dataset_name = "publication-venues"
release_id = latest_release
url = f"https://api.semanticscholar.org/datasets/v1/release/{release_id}/dataset/{dataset_name}"

headers = {"x-api-key": S2_API_KEY}
response = requests.get(url, headers=headers)
response

In [ ]:
response_publication_venues = s2_api.download_dataset(
    release_id=latest_release, dataset_name="publication-venues"
)

In [ ]:
url = response_publication_venues.json()["files"][0]
file_path = s2_api.download_file(url=url, path=release_path / "publication-venues")

In [ ]:
with gzip.open(
    file_path,
    "r",
) as f:
    file_content = f.read()

df_venues = pd.read_json(StringIO(file_content.decode()), lines=True)
publication_venue_ids = df_venues[df_venues["issn"].isin(issn)].id.unique()
print(f"{len(publication_venue_ids)} publication venue IDs")

In [ ]:
# df_venues.to_csv(f"publication-venues-{release_id}.csv", index=False)

## Papers

In [ ]:
response_papers = s2_api.download_dataset(release_id=latest_release, dataset_name="papers")
response_papers.status_code

In [ ]:
print(f"{len(response_papers.json()['files'])} files with papers")

In [ ]:
def download_paper_batch(
    url: str,
    publication_venue_ids: list,
    persistant: bool = False,
    start_date: str = "2023-01-01",
) -> None:
    file_name = url.split("?")[0].split("/")[-1]
    if (release_path / "papers" / str(file_name + ".csv")).exists():
        return

    file_path = download_file(url=url, path=release_path / "papers")
    file_content = ""
    with gzip.open(
        file_path,
        "r",
    ) as f:
        for line in tqdm(f.readlines(), leave=True):
            if any(x in line.decode() for x in publication_venue_ids):
                file_content += line.decode()

    df_papers = pd.read_json(StringIO(file_content), lines=True)

    date_filter = datetime.strptime(start_date, "%Y-%m-%d")
    df_papers["publicationdate"] = pd.to_datetime(df_papers["publicationdate"])
    df_papers = df_papers[df_papers["publicationdate"] >= date_filter]

    df_papers.to_csv(release_path / "papers" / str(file_name + ".csv"), index=False)

    if not persistant:
        os.remove(file_path)


def download_all_papers(
    file_list: list, publication_venue_ids: list, persistant: bool = False
) -> None:
    for url in tqdm(file_list, total=len(file_list)):
        download_paper_batch(
            url=url, publication_venue_ids=publication_venue_ids, persistant=persistant
        )

In [ ]:
s2_api.download_all_papers(
    file_list=response_papers.json()["files"],
    release_path=release_path,
    publication_venue_ids=list(publication_venue_ids),
)

## Bulk paper download

In [ ]:
paper_files = [x for x in (release_path / "papers").glob("*.csv")]
dfs = [pd.read_csv(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} papers in total")

In [ ]:
MIN_NUMBER_OF_CITATIONS: int = 2

df_all_papers_citations = df_all_papers[df_all_papers["citationcount"] > MIN_NUMBER_OF_CITATIONS]

In [ ]:
paper_ids = [x.split("/")[-1] for x in df_all_papers_citations["url"]]
print(f"{len(paper_ids):,} paper ids")

In [ ]:
json_path = release_path / "json"
json_path.mkdir(parents=True, exist_ok=True)

for i, group in tqdm(
    enumerate(s2_api.chunker(paper_ids[:2_000], MAX_NUM_BULK_PAPERS)),
    total=int(2_000 / MAX_NUM_BULK_PAPERS),
):
    response_batch = s2_api.download_bulk_paper(group)

    json_file = json_path / f"{release_id}-{i}.json"
    with open(str(json_file), "w", encoding="utf-8") as f:
        json.dump(response_batch.json(), f)

## Read json files

In [ ]:
paper_files = [x for x in (release_path / "json").glob("*.json")]
dfs = [pd.read_json(f) for f in paper_files]
df_all_papers = pd.concat(dfs)
print(f"{len(df_all_papers):,} json in total")

# Incremental Updates (WiP)

In [ ]:
response_releases = requests.get("https://api.semanticscholar.org/datasets/v1/release").json()
latest_release = response_releases[-1]
print(f"{response_releases[-1]} latest release")

In [ ]:
response_releases[-3:]

In [ ]:
release_path = RAW_DATA_DIR / "semantic_scholar" / "release" / f"{latest_release}"
release_path.mkdir(exist_ok=True, parents=True)

In [ ]:
from typing import Literal

URL_INCREMENTAL = "https://api.semanticscholar.org/datasets/v1/diffs/{start_release_id}/to/{end_release_id}/{dataset_name}"


def download_incremental(
    start_release_id: str,
    end_release_id: str,
    dataset_name: Literal[
        "abstracts",
        "authors",
        "citations",
        "embeddings-specter_v1",
        "embeddings-specter_v2",
        "paper-ids",
        "papers",
        "publication-venues",
        "s2orc",
        "s2orc_v2",
        "tldrs",
    ],
) -> requests.Response:
    headers = {"x-api-key": S2_API_KEY}
    response = requests.get(
        URL_INCREMENTAL.format(
            start_release_id=start_release_id,
            end_release_id=end_release_id,
            dataset_name=dataset_name,
        ),
        headers=headers,
        timeout=60,
    )
    return response


response = download_incremental(
    start_release_id="2026-01-27", end_release_id="2026-02-10", dataset_name="papers"
)

In [ ]:
# ['dataset', 'start_release', 'end_release', 'diffs']
len(response.json()["diffs"][0]["update_files"])

In [ ]:
response.json()["diffs"][0]["update_files"][0]